# Get the 1,300 GenImage test photos for RQ3 (run in Google Colab)

RQ3 needs the original GenImage photos of M1's test split (`m1_final_n500`): 300 from ADM and VQDM
(`test_seen`) and 1,000 from Stable Diffusion v1.4 (`test_unseen`). Kruthika's extracted folder
`GenImage_native_n500` is no longer available, so this notebook takes exactly those photos from the
**official GenImage archives**, the same archives that folder was built from.

It reads only the 1,300 photos out of the archives, without downloading or unzipping them, using the
archive reader from Kruthika's `M1_final_n500.ipynb` (Section 0.4). Every photo is checked against the
SHA-1 fingerprint M1 recorded for it, so these are byte-for-byte the photos M1 was tested on.

**Before running (5 minutes, once):**
1. Open the official GenImage Google Drive folder:
   https://drive.google.com/drive/folders/1jGt10bwTbhEZuGXLyvrCuxOI0cBqQ1FS
2. In your own Drive, make the folder `My Drive/Datasets/GenImage_official` (New > Folder).
3. In the official folder, right-click **ADM** > **Organize** > **Add shortcut** > choose
   `My Drive/Datasets/GenImage_official`. Do the same for **VQDM** and for the **Stable Diffusion v1.4**
   folder. Shortcuts use none of your Drive space.

**Then:** *Runtime > Run all* (no GPU needed). It takes about 10 to 30 minutes, mostly reading
from Drive, and writes `My Drive/genimage_test_photos.zip` (about 340 MB). Send that zip on; it is
added to the Kaggle dataset as `genimage/images/`. If Colab disconnects, run all again: photos already
written are skipped.

In [ ]:
ARCHIVE_SOURCE_DIR = '/content/drive/MyDrive/Datasets/GenImage_official'   # where the 3 shortcuts are
OUT_DIR = '/content/genimage_test'                                          # temporary, in Colab
OUT_ZIP = '/content/drive/MyDrive/genimage_test_photos.zip'                 # the result, in your Drive

import os, re, io, gzip, base64, hashlib, zipfile
from pathlib import Path
from collections import defaultdict
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
assert Path(ARCHIVE_SOURCE_DIR).is_dir(), f'Make the folder {ARCHIVE_SOURCE_DIR} and add the 3 shortcuts (see above).'
print('Visible in', ARCHIVE_SOURCE_DIR, ':', sorted(p.name for p in Path(ARCHIVE_SOURCE_DIR).iterdir()))

## The 1,300 photos
From M1's split files `test_seen.csv` and `test_unseen.csv` (`m1_final_n500`): each photo's path inside
GenImage, its SHA-1 and its size. Stored here so the notebook needs no other file.

In [ ]:
TEST_LIST_B64 = '...'
wanted = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(TEST_LIST_B64))))
wanted['generator'] = wanted['image_id'].str.split('/').str[0]
wanted['member'] = wanted['image_id'].str.split('/', n=1).str[1]           # path inside that generator's archive
print(len(wanted), 'photos:', wanted['generator'].value_counts().to_dict())

## Archive reader (from Kruthika's `M1_final_n500.ipynb`, Section 0.4, unchanged)
Reads single files out of a split zip (`name.z01`, `name.z02`, ..., `name.zip`) without joining the
parts: the last part holds the table of contents, which says where each file sits.

In [ ]:
IMAGE_EXTS_S0 = {'.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tif', '.tiff'}
ARCHIVE_RE = re.compile(r'\.(zip|z\d{2}|\d{3}|tar|tgz|gz|bz2|xz|rar|7z)$', re.IGNORECASE)

import struct, zlib, bisect
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm


class Member:
    """One file inside an archive: its path in the archive, sizes, and where its data starts."""
    def __init__(self, filename, file_size, compress_size, method, crc, flags, disk, offset):
        self.filename, self.file_size, self.compress_size = filename, file_size, compress_size
        self.method, self.crc, self.flags, self.disk, self.offset = method, crc, flags, disk, offset


class SplitZip:
    """Read single files out of a SPLIT zip (name.z01, name.z02, ..., name.zip) WITHOUT joining the parts.

    How: the last part (.zip) ends with the archive's table of contents (the "central directory").
    For every file it stores (part number, byte offset in that part). We read the table once,
    then jump straight to the few images we want. Only those bytes are read from Drive.
    """
    def __init__(self, parts):
        self.parts = [str(p) for p in parts]                      # in order: .z01, .z02, ..., .zip
        sizes = [os.path.getsize(p) for p in self.parts]
        self.starts = [0]
        for s in sizes:
            self.starts.append(self.starts[-1] + s)               # where each part begins in the joined stream
        self.total = self.starts[-1]
        self._read_central_directory()

    def _read(self, pos, n):
        """Read n bytes at position pos of the (virtual) joined stream; may cross part boundaries."""
        out = bytearray()
        while n > 0:
            i = bisect.bisect_right(self.starts, pos) - 1
            with open(self.parts[i], "rb") as f:
                f.seek(pos - self.starts[i])
                chunk = f.read(min(n, self.starts[i + 1] - pos))
            if not chunk:
                raise IOError("unexpected end of archive")
            out += chunk; pos += len(chunk); n -= len(chunk)
        return bytes(out)

    def _abs(self, disk, offset):
        return self.starts[disk] + offset if disk < len(self.parts) else -1

    def _read_central_directory(self):
        tail_len = min(self.total, 65557 + 20)
        tail = self._read(self.total - tail_len, tail_len)
        e = tail.rfind(b"PK\x05\x06")                              # "end of central directory" record
        if e < 0:
            raise ValueError("no zip end record found: is the last part (.zip) complete?")
        this_disk, cd_disk, _, n_entries, cd_size, cd_off = struct.unpack("<HHHHII", tail[e + 4:e + 20])
        loc = tail.rfind(b"PK\x06\x07", 0, e)                       # Zip64 locator (archives > 65535 files or > 4 GB)
        if loc >= 0:
            z_disk, z_off, _ = struct.unpack("<IQI", tail[loc + 4:loc + 20])
            rec = self._read(self._abs(z_disk, z_off), 56)
            if rec[:4] != b"PK\x06\x06":
                raise ValueError("broken Zip64 record")
            this_disk, cd_disk, _, n_entries, cd_size, cd_off = struct.unpack("<IIQQQQ", rec[16:56])
        if this_disk not in (0, len(self.parts) - 1):
            raise ValueError(f"the archive says it has {this_disk + 1} parts, but {len(self.parts)} were found: a part is missing")
        cd = self._read(self._abs(cd_disk, cd_off), cd_size)
        self.members, p = [], 0
        while p + 46 <= len(cd) and cd[p:p + 4] == b"PK\x01\x02":
            (_, _, _, flags, method, _, _, crc, csize, usize, nlen, elen, clen,
             disk, _, _, off) = struct.unpack("<IHHHHHHIIIHHHHHII", cd[p:p + 46])
            raw = cd[p + 46:p + 46 + nlen]
            name = raw.decode("utf-8" if flags & 0x800 else "cp437")
            extra = cd[p + 46 + nlen:p + 46 + nlen + elen]
            q = 0
            while q + 4 <= len(extra):                             # Zip64 extra field holds the large values
                hid, hlen = struct.unpack("<HH", extra[q:q + 4])
                if hid == 1:
                    vals, r = extra[q + 4:q + 4 + hlen], 0
                    if usize == 0xFFFFFFFF: usize = struct.unpack("<Q", vals[r:r + 8])[0]; r += 8
                    if csize == 0xFFFFFFFF: csize = struct.unpack("<Q", vals[r:r + 8])[0]; r += 8
                    if off == 0xFFFFFFFF:   off = struct.unpack("<Q", vals[r:r + 8])[0]; r += 8
                    if disk == 0xFFFF:      disk = struct.unpack("<I", vals[r:r + 4])[0]
                q += 4 + hlen
            self.members.append(Member(name, usize, csize, method, crc, flags, disk, off))
            p += 46 + nlen + elen + clen
        if len(self.members) != n_entries:
            raise ValueError(f"table of contents lists {n_entries} files but {len(self.members)} were read")

    def read(self, m):
        if m.flags & 0x1:
            raise ValueError("file is encrypted")
        start = self._abs(m.disk, m.offset)
        head = self._read(start, 30)                               # local file header
        if head[:4] != b"PK\x03\x04":
            raise ValueError(f"bad local header for {m.filename}")
        nlen, elen = struct.unpack("<HH", head[26:30])
        data = self._read(start + 30 + nlen + elen, m.compress_size)
        if m.method == 8:
            data = zlib.decompressobj(-15).decompress(data)        # deflate
        elif m.method != 0:                                         # 0 = stored
            raise ValueError(f"compression method {m.method} not supported here; use the manual steps in 0.3")
        if zlib.crc32(data) & 0xFFFFFFFF != m.crc or len(data) != m.file_size:
            raise ValueError(f"checksum mismatch for {m.filename}")
        return data


def list_archives(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.is_file() and ARCHIVE_RE.search(p.name))


def group_zip_sets(arcs):
    """Group files into zip archives: name.zip (+ name.z01, name.z02 ...) or name.zip.001, .002 ...
    Returns ([(set_name, [parts in order])], [problems])."""
    sets, problems, used = [], [], set()
    for a in arcs:
        if a.suffix.lower() == ".zip":                             # split style: .z01 ... .zNN come before .zip
            zs = sorted((p for p in arcs if p.parent == a.parent and p.stem == a.stem
                         and re.fullmatch(r"\.z\d{2}", p.suffix, re.IGNORECASE)), key=lambda p: int(p.suffix[2:]))
            nums = [int(p.suffix[2:]) for p in zs]
            if nums != list(range(1, len(nums) + 1)):
                gaps = [f"{a.stem}.z{n:02d}" for n in range(1, max(nums) + 1) if n not in nums]
                problems.append(f"{a.name}: missing part(s) {gaps}; copy/shortcut them too, then rerun 0.4")
            else:
                sets.append((a.name, zs + [a]))
            used.update(zs + [a])
    numbered = defaultdict(list)                                   # byte-split style: name.zip.001, name.zip.002 ...
    for a in arcs:
        if re.fullmatch(r"\.\d{3}", a.suffix) and a.stem.lower().endswith(".zip"):
            numbered[(a.parent, a.stem)].append(a); used.add(a)
    for (_, stem), ps in numbered.items():
        sets.append((stem, sorted(ps, key=lambda p: int(p.suffix[1:]))))
    leftover = defaultdict(list)
    for a in arcs:
        if a not in used:
            leftover[a.stem].append(a.name)
    for stem, names in leftover.items():
        problems.append(f"{names[:3]}{' ...' if len(names) > 3 else ''}: not a zip archive, or its final '{stem}.zip' "
                        "part is missing; use the manual steps in 0.3")
    return sets, problems


def safe_member(name):
    p = Path(name)
    return not p.is_absolute() and ".." not in p.parts          # never write outside DATASET_DIR

## Find the three generator folders

In [ ]:
PATTERNS = {'ADM': r'adm', 'VQDM': r'vqdm', 'stable_diffusion_v_1_4': r'(stablediffusion|sd)v?14'}   # as in M1 Section 0.2b (matched anywhere in the name)
norm = lambda name: re.sub(r'[^a-z0-9]', '', name.lower())
folders = {}
for gen, pat in PATTERNS.items():
    hits = [p for p in Path(ARCHIVE_SOURCE_DIR).iterdir() if p.is_dir() and re.search(pat, norm(p.name))]
    if len(hits) != 1:
        raise SystemExit(f'Expected one {gen} folder in {ARCHIVE_SOURCE_DIR}, found {[h.name for h in hits]}. '
                         'Add its shortcut from the official GenImage folder (see the top of this notebook).')
    folders[gen] = hits[0]
    print(f'{gen:<24} -> {hits[0].name}')

## Read the 1,300 photos and check each one

In [ ]:
out = Path(OUT_DIR)
status = {}                                                    # image_id -> 'ok' or a problem
def done(r):
    p = out / r.image_id
    return p.exists() and hashlib.sha1(p.read_bytes()).hexdigest() == r.file_hash

for gen, folder in folders.items():
    need = wanted[wanted.generator == gen]
    todo = {r.member: r for r in need.itertuples() if not done(r)}
    for r in need.itertuples():
        if r.member not in todo:
            status[r.image_id] = 'ok'
    sets, problems = group_zip_sets(list_archives(folder))
    for p in problems:
        print('PROBLEM', gen, p)
    print(f'\n{gen}: {len(need)} photos, {len(todo)} still to read, {len(sets)} archive(s)')
    for name, parts in sets:
        if not todo:
            break
        zf = SplitZip(parts)
        by_name = {m.filename: m for m in zf.members}
        found = [(todo[k], by_name[k]) for k in list(todo) if k in by_name]
        print(f'  {name}: {len(zf.members):,} files in the archive, {len(found)} of ours')

        def copy_one(item):
            r, m = item
            data = zf.read(m)
            if hashlib.sha1(data).hexdigest() != r.file_hash:
                return r, 'SHA-1 differs from M1'
            target = out / r.image_id
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
            return r, 'ok'

        with ThreadPoolExecutor(8) as pool:
            for r, s in tqdm(pool.map(copy_one, found), total=len(found), leave=False):
                status[r.image_id] = s
                todo.pop(r.member, None)
    for r in todo.values():
        status[r.image_id] = 'not found in the archives'

report = pd.Series(status).reindex(wanted.image_id).fillna('not read')
print('\n', report.value_counts().to_string())
bad = report[report != 'ok']
if len(bad):
    print('\nPROBLEMS (first 10):'); print(bad.head(10).to_string())
else:
    print('\nAll 1,300 photos read, and every one matches M1\'s SHA-1.')

## Write the zip to your Drive

In [ ]:
with zipfile.ZipFile(OUT_ZIP, 'w', zipfile.ZIP_STORED) as z:        # photos are already compressed
    for r in wanted.itertuples():
        p = out / r.image_id
        if p.exists():
            z.write(p, f'genimage/images/{r.image_id}')
    z.writestr('genimage/images/CHECK.txt', report.value_counts().to_string() + '\n')
print(f'Wrote {OUT_ZIP} ({os.path.getsize(OUT_ZIP) / 1e6:.0f} MB). Download it from your Drive and send it on.')